# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
record_sets = dataset.record_sets
print('Available record sets:')
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")

# For demonstration, select the first record set
if record_sets:
    selected_record_set_id = record_sets[0]['@id']
    print(f"\nFields for record set '{selected_record_set_id}':")
    fields = record_sets[0].get('field', [])
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"  - {field_id}")

# Show a sample of the records using the record set @id
print(f"\nSample records from record set '{selected_record_set_id}':")
for i, rec in enumerate(dataset.records(record_set=selected_record_set_id)):
    print(rec)
    if i >= 2:
        break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.
Below we demonstrate for each record set by their `@id`.

In [ ]:
# Collect all the record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Display columns for the primary record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in record set '{main_rs_id}':")
    if main_rs_id in dataframes:
        print(dataframes[main_rs_id].columns.tolist())
        dataframes[main_rs_id].head()
    else:
        print("No records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Filter records for a numeric mental health assessment (e.g., GAD-7 score field).
- Normalize it.
- Group by a key demographic variable (e.g., gender).

> All references are made using the `@id`.

In [ ]:
# Pick the main record set ID
record_set_id = main_rs_id
df = dataframes.get(record_set_id)

if df is not None and not df.empty:
    # Possible field @id's based on survey dataset
    # (Confirm by listing columns if necessary)
    print(df.columns)
    
    # Let's assume GAD-7 score column has @id 'http://senscience.ai/gad7_score'
    numeric_field_id = 'http://senscience.ai/gad7_score' if 'http://senscience.ai/gad7_score' in df.columns else df.columns[-1]
    
    # Set a threshold for filtering
    threshold = 10
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        # Filter records
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by gender field (e.g., @id 'http://senscience.ai/gender')
        group_field_id = 'http://senscience.ai/gender'
        if group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped average {numeric_field_id} by {group_field_id}:")
            print(grouped_df)
    else:
        print(f"Field '{numeric_field_id}' is not numeric. Please check field names and types.")
else:
    print("No data loaded for the main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using the `@id` references for columns.

In [ ]:
# Example: Visualize GAD-7 score distribution and by gender
if df is not None and not df.empty and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, color='skyblue')
    plt.title('Distribution of GAD-7 Score')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by gender
    if group_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides rich demographic and psychological data for residents of Kilifi County, Kenya.
- Data loading and processing is facilitated by the Croissant schema and the `mlcroissant` library.
- We explored numeric scores (e.g., GAD-7), normalized them, and visualized distributions and group differences by gender (using `@id`).
- Further analyses could include trend analysis, more complex cohort selection, or integration with other datasets.

For any detailed machine learning or statistical analysis, please ensure a thorough review of field @id mappings and the Croissant schema documentation for this dataset.